In [2]:
import pandas as pd

# Review of initial Txt2Onto human host prediction from medatadata validation results

In [3]:
#Create dataframe of validation scores
validation = pd.read_parquet(r"../artifacts_updated/stage1/val_scores.parquet")

In [4]:
validation.head()

,accession,text_hash,label,p_human
0,SAMN52580513,4df1bf776b39a6be7a5cf2f5a8ccce569c71d104,1,0.989664
1,SAMEA120306985,c09b26705117b6cec5d3d09458623142b28d9cd3,1,0.000542
2,SAMN45458553,09d43f3df711151ed1e636c977d751f3cb21bd92,1,0.974748
3,SAMN45458554,ea50336a5161b9592a808160307f096abaf6e568,1,0.980346
4,SAMN45458555,6be7a60ab65a2509515de969f30a9196221e8ea3,1,0.975322


In [5]:
len(validation)

126565

In [6]:
#Creating dataframe of original dataset with accompanying metadata
original_df = pd.read_parquet(r"../data/human_labeled.parquet")

In [7]:
len(original_df)

1511070

In [8]:
original_df.head()

,Unnamed: 0,biosample_accession,sample_taxon_id,sample_taxonomy_name,host_comments,host_name,isolation_comments,comments,category
0,1176,SAMN52580513,2890317,Yersinia alsatica,Not available,Homo sapiens,PHE | human,NaN,1
1,4309,SAMN52583663,1314,Streptococcus pyogenes,NaN,Homo sapiens,blood,NaN,1
2,4310,SAMN52583664,1314,Streptococcus pyogenes,NaN,Homo sapiens,wound,NaN,1
3,4314,SAMN52583668,485,Neisseria gonorrhoeae,NaN,Homo sapiens,upper genitourinary tract,NaN,1
4,4315,SAMN52583669,487,Neisseria meningitidis,NaN,Homo sapiens,throat,NaN,1


In [9]:
#Merging dataframes on biosample accession 
validation_df = pd.merge(validation, original_df, how='inner', left_on = 'accession', right_on='biosample_accession')
len(validation_df)

126565

In [10]:
validation_df.columns

Index(['accession', 'text_hash', 'label', 'p_human', 'Unnamed: 0',
       'biosample_accession', 'sample_taxon_id', 'sample_taxonomy_name',
       'host_comments', 'host_name', 'isolation_comments', 'comments',
       'category'],
      dtype='str')

In [11]:
#drop columns irrelevant to review
#'label' is redundant since it is the same information as 'category' from the original dataset
validation_df = validation_df.drop(['text_hash', 'Unnamed: 0', 'label'], axis=1)

In [12]:
import numpy as np

#Assign human/not human labels
validation_df['category'] = validation_df['category'].replace({1: 'human', -1: 'not human'})

In [13]:
validation_df['category'].unique()

array(['human', 'not human'], dtype=object)

In [14]:
validation_df.head()

,accession,p_human,biosample_accession,sample_taxon_id,sample_taxonomy_name,host_comments,host_name,isolation_comments,comments,category
0,SAMN52580513,0.989664,SAMN52580513,2890317,Yersinia alsatica,Not available,Homo sapiens,PHE | human,NaN,human
1,SAMEA120306985,0.000542,SAMEA120306985,28901,Salmonella enterica,healthy,Homo sapiens,NA | aquarium water,NaN,human
2,SAMN45458553,0.974748,SAMN45458553,548,Klebsiella aerogenes,NaN,Homo sapiens,IDR2400068773-01-02 | rectal swab | whole orga...,NaN,human
3,SAMN45458554,0.980346,SAMN45458554,573,Klebsiella pneumoniae,NaN,Homo sapiens,IDR2400068904-01-02 | rectal swab | whole orga...,NaN,human
4,SAMN45458555,0.975322,SAMN45458555,470,Acinetobacter baumannii,NaN,Homo sapiens,IDR2400069003-01-02 | rectal swab | whole orga...,NaN,human


In [15]:
validation_df.to_parquet(r"../data exploration/validation_review.parquet")

In [89]:
validation_label = validation_df['p_human'] > .5

validation_df['validation_label'] = validation_label

validation_df['validation_label'] = validation_df['validation_label'].replace({True: 'human', False: 'not human'})

In [16]:
print(validation_df['category'].value_counts())

category
human        95431
not human    31134
Name: count, dtype: int64


In [90]:
mislabeled_mask = ~(validation_df['validation_label'] == validation_df['category'])
print(sum(mislabeled_mask))
mislabeled = validation_df[mislabeled_mask]

9061


In [95]:
mislabeled[mislabeled['category'] == 'not human'].sample(20)

,accession,label,p_human,biosample_accession,sample_taxon_id,sample_taxonomy_name,host_comments,host_name,isolation_comments,comments,category,validation_label
108881,SAMN09866910,0,0.709643,SAMN09866910,562,Escherichia coli,NaN,Chroicocephalus,478 | ESBA | Rectal swab,NaN,not human,human
103891,SAMN43271811,0,0.937651,SAMN43271811,222,Achromobacter,NaN,Achromobacter sp. As-55,spontaneous mutant SMAs-55 under the exposure ...,NaN,not human,human
121772,SAMN38263270,0,0.562821,SAMN38263270,287,Pseudomonas aeruginosa,NaN,Mus musculus,OCE1P4622 | lung | Single bacterial colony,NaN,not human,human
107177,SAMEA115457951,0,0.570614,SAMEA115457951,562,Escherichia coli,healthy,chicken,B23CTX | feces,NaN,not human,human
101119,SAMN25249711,0,0.999695,SAMN25249711,287,Pseudomonas aeruginosa,Cystic Fibrosis,CF-006,CF06.1.08 | University of Texas Southwestern M...,NaN,not human,human
109969,SAMN40372511,0,0.751978,SAMN40372511,562,Escherichia coli,not collected,Wild boar,Ec30 | University of Sassari | not collected,NaN,not human,human
125259,SAMEA114440218,0,0.973186,SAMEA114440218,90370,Salmonella enterica subsp. enterica serovar Typhi,not provided,Salmonella enterica subsp. enterica serovar Typhi,S265 | blood,NaN,not human,human
102261,SAMN43048240,0,0.645222,SAMN43048240,287,Pseudomonas aeruginosa,NaN,lab culture in SCFM2,SCFM2_CTevolved_planktonicPOP1 | PAO1 evolved ...,NaN,not human,human
117100,SAMN38884008,0,0.739495,SAMN38884008,573,Klebsiella pneumoniae,NaN,horse,ANSES | respiratory,NaN,not human,human
109194,SAMN16213908,0,0.732278,SAMN16213908,28903,Mycoplasmopsis bovis,NaN,Bos taurus,M0023 | whole organism,NaN,not human,human
